In [2]:
# Medicare Provider Analytics Using Apache Spark

## Data Cleaning and Preparation

# Course:CS-675 Big Data Management & Analytics  
# Author: Judi-Ann Beckford  

### Objective

# Clean and prepare the Medicare Provider Service and Public Provider Enrollment datasets for integration using the National Provider Identifier (NPI).

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

In [4]:
spark = (
    SparkSession.builder
    .appName("Medicare Data Cleaning")
    .config("spark.sql.shuffle.partitions", "24")
    .config("spark.default.parallelism", "24")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Driver memory:", spark.sparkContext.getConf().get("spark.driver.memory"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
26/07/28 21:08:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/28 21:08:45 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.1.2
Spark master: local[6]
Driver memory: 6g


In [5]:
provider_file = "../data/raw/PHY_R26_P05_V10_D24_Prov_Svc.csv"
enrollment_file = "../data/raw/PPEF_Enrollment_Extract_2026.04.01.csv"

print("Provider file exists:", os.path.exists(provider_file))
print("Enrollment file exists:", os.path.exists(enrollment_file))
print("Current folder:", os.getcwd())

Provider file exists: True
Enrollment file exists: True
Current folder: /Users/judi-annbeckford/Documents/Medicare-Provider-Analytics-Spark/notebooks


In [6]:
provider_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(provider_file)
)

enrollment_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(enrollment_file)
)

print("Both raw datasets loaded.")

[Stage 3:==========================================>              (18 + 6) / 24]

Both raw datasets loaded.


In [7]:
provider_clean_df = (
    provider_raw_df
    .select(
        F.col("Rndrng_NPI").cast("long").alias("NPI"),
        F.trim(F.col("Rndrng_Prvdr_Type")).alias("PROVIDER_SPECIALTY"),
        F.upper(F.trim(F.col("Rndrng_Prvdr_State_Abrvtn"))).alias("PROVIDER_STATE"),
        F.trim(F.col("HCPCS_Cd")).alias("HCPCS_CODE"),
        F.trim(F.col("HCPCS_Desc")).alias("HCPCS_DESCRIPTION"),
        F.trim(F.col("Place_Of_Srvc")).alias("PLACE_OF_SERVICE"),
        F.col("Tot_Benes").cast("double").alias("TOTAL_BENEFICIARIES"),
        F.col("Tot_Srvcs").cast("double").alias("TOTAL_SERVICES"),
        F.col("Avg_Sbmtd_Chrg").cast("double").alias("AVG_SUBMITTED_CHARGE"),
        F.col("Avg_Mdcr_Alowd_Amt").cast("double").alias("AVG_MEDICARE_ALLOWED"),
        F.col("Avg_Mdcr_Pymt_Amt").cast("double").alias("AVG_MEDICARE_PAYMENT"),
        F.col("Avg_Mdcr_Stdzd_Amt").cast("double").alias("AVG_STANDARDIZED_PAYMENT")
    )
    .filter(F.col("NPI").isNotNull())
    .filter(F.col("HCPCS_CODE").isNotNull())
)

In [8]:
enrollment_clean_df = (
    enrollment_raw_df
    .select(
        F.col("NPI").cast("long").alias("NPI"),
        F.trim(F.col("PROVIDER_TYPE_DESC")).alias("ENROLLMENT_PROVIDER_TYPE"),
        F.upper(F.trim(F.col("STATE_CD"))).alias("ENROLLMENT_STATE"),
        F.trim(F.col("FIRST_NAME")).alias("FIRST_NAME"),
        F.trim(F.col("MDL_NAME")).alias("MIDDLE_NAME"),
        F.trim(F.col("LAST_NAME")).alias("LAST_NAME"),
        F.trim(F.col("ORG_NAME")).alias("ORGANIZATION_NAME")
    )
    .filter(F.col("NPI").isNotNull())
)

In [9]:
print("CLEAN PROVIDER-SERVICE SCHEMA")
provider_clean_df.printSchema()

print("\nCLEAN ENROLLMENT SCHEMA")
enrollment_clean_df.printSchema()

CLEAN PROVIDER-SERVICE SCHEMA
root
 |-- NPI: long (nullable = true)
 |-- PROVIDER_SPECIALTY: string (nullable = true)
 |-- PROVIDER_STATE: string (nullable = true)
 |-- HCPCS_CODE: string (nullable = true)
 |-- HCPCS_DESCRIPTION: string (nullable = true)
 |-- PLACE_OF_SERVICE: string (nullable = true)
 |-- TOTAL_BENEFICIARIES: double (nullable = true)
 |-- TOTAL_SERVICES: double (nullable = true)
 |-- AVG_SUBMITTED_CHARGE: double (nullable = true)
 |-- AVG_MEDICARE_ALLOWED: double (nullable = true)
 |-- AVG_MEDICARE_PAYMENT: double (nullable = true)
 |-- AVG_STANDARDIZED_PAYMENT: double (nullable = true)


CLEAN ENROLLMENT SCHEMA
root
 |-- NPI: long (nullable = true)
 |-- ENROLLMENT_PROVIDER_TYPE: string (nullable = true)
 |-- ENROLLMENT_STATE: string (nullable = true)
 |-- FIRST_NAME: string (nullable = true)
 |-- MIDDLE_NAME: string (nullable = true)
 |-- LAST_NAME: string (nullable = true)
 |-- ORGANIZATION_NAME: string (nullable = true)



In [10]:
provider_clean_df.show(5, truncate=False)

[Stage 4:>                                                          (0 + 1) / 1]

+----------+------------------+--------------+----------+----------------------------------------------------------------------------------------------------------------------------------+----------------+-------------------+--------------+--------------------+--------------------+--------------------+------------------------+
|NPI       |PROVIDER_SPECIALTY|PROVIDER_STATE|HCPCS_CODE|HCPCS_DESCRIPTION                                                                                                                 |PLACE_OF_SERVICE|TOTAL_BENEFICIARIES|TOTAL_SERVICES|AVG_SUBMITTED_CHARGE|AVG_MEDICARE_ALLOWED|AVG_MEDICARE_PAYMENT|AVG_STANDARDIZED_PAYMENT|
+----------+------------------+--------------+----------+----------------------------------------------------------------------------------------------------------------------------------+----------------+-------------------+--------------+--------------------+--------------------+--------------------+------------------------+
|1003000126|I

In [11]:
enrollment_clean_df.show(5, truncate=False)

+----------+--------------------------------+----------------+----------+-----------+---------+-----------------+
|NPI       |ENROLLMENT_PROVIDER_TYPE        |ENROLLMENT_STATE|FIRST_NAME|MIDDLE_NAME|LAST_NAME|ORGANIZATION_NAME|
+----------+--------------------------------+----------------+----------+-----------+---------+-----------------+
|1003000126|PRACTITIONER - INTERNAL MEDICINE|MD              |ARDALAN   |NULL       |ENKESHAFI|NULL             |
|1003000126|PRACTITIONER - HOSPITALIST      |DC              |ARDALAN   |NULL       |ENKESHAFI|NULL             |
|1003000126|PRACTITIONER - INTERNAL MEDICINE|VA              |ARDALAN   |NULL       |ENKESHAFI|NULL             |
|1003000126|PRACTITIONER - INTERNAL MEDICINE|PA              |ARDALAN   |NULL       |ENKESHAFI|NULL             |
|1003000134|PRACTITIONER - PATHOLOGY        |IL              |THOMAS    |L          |CIBULL   |NULL             |
+----------+--------------------------------+----------------+----------+-----------+---

In [ ]:
## Cleaning Completed So Far

# - Selected only the variables required for the analytical questions.
# - Standardized both NPI fields as long integers.
# - Standardized state abbreviations using uppercase values.
# - Trimmed unnecessary whitespace from categorical variables.
# - Converted service, beneficiary, charge, and payment variables to numeric types.
# - Removed records without an NPI.
# - Removed provider-service records without an HCPCS procedure code.
# - Retained enrollment duplicates temporarily to prevent accidental loss of legitimate provider records.